# ME 313 · Lab 4.2 — PINN a cooling wall  ⭐ *showpiece*
**Module 4 companion (Transient Conduction).**

A physics-informed neural network (PINN) solves the **time-dependent** heat equation directly from its residual — no mesh, no time-stepping. 
Here we solve a **symmetric plane wall cooling by convection** and compare the PINN's centre-temperature history to the Module 4 **one-term series** solution $\theta^*_o=C_1e^{-\zeta_1^2 Fo}$. 

> **Runtime note:** the training cell uses PyTorch and is meant to run on **Google Colab** (CPU is fine; a GPU is faster). Run cells top to bottom.  
> If `torch` is missing, run `!pip install torch` in a cell first.


### 1. Problem & the classical reference
Nondimensional plane wall (half-thickness $L$), symmetric convective cooling, Biot $Bi=1$. 
Variables: $\xi=x/L\in[0,1]$ (0 = midplane, 1 = surface), $Fo=\alpha t/L^2$, $\theta^*=(T-T_\infty)/(T_i-T_\infty)$. 
The PDE is $\partial\theta^*/\partial Fo=\partial^2\theta^*/\partial\xi^2$, with symmetry $\theta^*_\xi(0,Fo)=0$, convection $-\theta^*_\xi(1,Fo)=Bi\,\theta^*(1,Fo)$, and $\theta^*(\xi,0)=1$.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
Bi = 1.0
# Module 4 one-term reference (plane wall, Bi=1): zeta1, C1 from the table
zeta1, C1 = 0.8603, 1.1191
def theta_o_oneterm(Fo):
    return C1*np.exp(-zeta1**2 * Fo)   # centreline theta*, valid Fo > 0.2
print('reference ready (one-term, Bi=1): theta*_o(Fo=0.5) =', round(float(theta_o_oneterm(0.5)),4))

### 2. The PINN
A small network $\hat\theta(\xi,Fo)$ trained to drive the PDE residual plus the initial and boundary conditions to zero.


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1))

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

Fo_max = 1.0
def sample(n):
    xi = torch.rand(n,1); Fo = torch.rand(n,1)*Fo_max
    xF = torch.cat([xi, Fo], 1).requires_grad_(True)
    return xF

def loss_fn():
    # interior PDE residual
    xF = sample(2000); th = net(xF)
    d = grad(th, xF); th_xi, th_Fo = d[:,0:1], d[:,1:2]
    th_xixi = grad(th_xi, xF)[:,0:1]
    r_pde = th_Fo - th_xixi
    # initial condition theta*=1 at Fo=0
    xi0 = torch.rand(400,1); ic = torch.cat([xi0, torch.zeros_like(xi0)],1)
    r_ic = net(ic) - 1.0
    # symmetry at xi=0 : d theta/d xi = 0
    Fb = torch.rand(400,1)*Fo_max
    xs = torch.cat([torch.zeros_like(Fb), Fb],1).requires_grad_(True)
    r_sym = grad(net(xs), xs)[:,0:1]
    # convection at xi=1 : -d theta/d xi = Bi*theta
    xw = torch.cat([torch.ones_like(Fb), Fb],1).requires_grad_(True)
    thw = net(xw); thw_xi = grad(thw, xw)[:,0:1]
    r_bc = -thw_xi - Bi*thw
    mse = nn.MSELoss()
    z = lambda r: mse(r, torch.zeros_like(r))
    return z(r_pde) + z(r_ic) + z(r_sym) + z(r_bc)

opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for it in range(4000):
    opt.zero_grad(); L = loss_fn(); L.backward(); opt.step()
    if it % 800 == 0: print(f'iter {it:4d}  loss {L.item():.2e}')

### 3. Compare the PINN to the one-term solution
Read the PINN's centre temperature ($\xi=0$) over time and overlay the Module 4 curve (valid for $Fo>0.2$).


In [ ]:
Fo_grid = np.linspace(0.0, 1.0, 60)
with torch.no_grad():
    inp = torch.tensor(np.column_stack([np.zeros_like(Fo_grid), Fo_grid]), dtype=torch.float32)
    th_pinn = net(inp).numpy().ravel()

Fo_ot = Fo_grid[Fo_grid >= 0.2]
th_ot = theta_o_oneterm(Fo_ot)

plt.figure(figsize=(6,4))
plt.plot(Fo_grid, th_pinn, label='PINN (centre)')
plt.plot(Fo_ot, th_ot, '--', label='one-term (Fo>0.2)')
plt.xlabel('Fourier number  Fo'); plt.ylabel(r'$\theta^*_o$ (centre)')
plt.legend(); plt.grid(alpha=.3); plt.title('PINN vs. one-term series, Bi=1'); plt.show()

# quantitative check in the one-term-valid window
err = np.max(np.abs(np.interp(Fo_ot, Fo_grid, th_pinn) - th_ot))
print(f'max |PINN - one-term| for Fo>0.2:  {err:.3f}')

### 4. Reflect & extend
- The PINN learned the full $\theta^*(\xi,Fo)$ field from the **equation alone** — no labelled data — and should track the one-term curve closely for $Fo>0.2$, while also capturing the early-time behaviour ($Fo<0.2$) where the one-term formula is *invalid*.
- **Verify:** at $Fo=0.5$ the one-term centre value is $\approx C_1e^{-\zeta_1^2(0.5)}$. Does the PINN agree?
- **Extend 1:** change `Bi` to 5 (update `zeta1, C1` from the table) and retrain.
- **Extend 2 (stretch):** add a moving heat source term to the residual to model a welding/AM pass.
- **AI companion:** ask an LLM to write the one-term solution for your $Bi$, then check its $C_1,\zeta_1$ against the Module 4 table before trusting the comparison.
